In [1]:
# %pip install -q minigrid gymnasium transformers accelerate torch sentencepiece

In [2]:
# %pip install --upgrade torch torchvision torchaudio

In [3]:
import torch
print(torch.__version__)

2.10.0+cu128


In [4]:
import json
from pathlib import Path
import time
from datetime import datetime, timezone
from collections import deque

import matplotlib.pyplot as plt
import pandas as pd
import gymnasium as gym
from minigrid.core.actions import Actions
from minigrid.wrappers import FullyObsWrapper
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM


In [ ]:
MODEL_CANDIDATES = [
    "Qwen/Qwen2.5-3B-Instruct",
    "Qwen/Qwen2.5-7B-Instruct",
]
MAX_NEW_TOKENS = 16
TEMPERATURE = 0.0
MOCK_MODE = False
HISTORY_WINDOW = 3
ENVIRONMENTS = [
    "MiniGrid-LavaGapS7-v0",
    "MiniGrid-BlockedUnlockPickup-v0",
    "MiniGrid-DoorKey-5x5-v0",
    "MiniGrid-Empty-8x8-v0",
]
N_EPISODES_LIST = (10,)
MAX_STEPS_LIST = (50,)
BUFFER_SIZES = (2, 5)
OUTPUT_ROOT = "CS 228/outputs"
RUN_TAG = "bot_sweep"


In [6]:
from dataclasses import dataclass

@dataclass
class ThoughtTemplate:
    name: str
    description: str
    reasoning_pattern: str
    usage_count: int = 0
    success_rate: float = 0.0

ALL_TEMPLATES = [
    ThoughtTemplate(
        name="Direct Navigation",
        description="Moving towards a visible target in an empty space.",
        reasoning_pattern="1. Locate target coordinates. 2. Identify relative direction. 3. Align orientation. 4. Move forward.",
    ),
    ThoughtTemplate(
        name="Obstacle Avoidance",
        description="Navigating around walls, lava, or closed doors.",
        reasoning_pattern="1. Identify blocking object. 2. Scan for open path. 3. Plan detour. 4. Execute lateral move.",
    ),
    ThoughtTemplate(
        name="Object Acquisition",
        description="Locating and picking up a key or other inventory item.",
        reasoning_pattern="1. Scan grid for target object (key/ball). 2. Navigate to the object. 3. Face the object. 4. Use pickup action.",
    ),
    ThoughtTemplate(
        name="Unlock Door",
        description="Using a held key to open a locked door.",
        reasoning_pattern="1. Confirm key is in inventory. 2. Navigate to the locked door. 3. Face the door. 4. Use toggle action to unlock and open.",
    ),
    ThoughtTemplate(
        name="Clear Path",
        description="Moving blocking objects out of the way to reach a target.",
        reasoning_pattern="1. Identify the blocking object. 2. Face the blocker. 3. Pickup or push the blocker aside. 4. Drop it in a clear cell. 5. Return to original objective.",
    ),
]

class MetaBuffer:
    def __init__(self, buffer_size=2):
        self.templates = {}
        self._initialize_templates(buffer_size)

    def _initialize_templates(self, buffer_size):
        for t in ALL_TEMPLATES[:buffer_size]:
            self.add_template(t)

    def add_template(self, template: ThoughtTemplate):
        self.templates[template.name] = template

    def retrieve(self, problem_description: str) -> ThoughtTemplate:
        desc = problem_description.lower()
        if "Clear Path" in self.templates and ("box" in desc or "ball" in desc):
            return self.templates["Clear Path"]
        if "Object Acquisition" in self.templates and "key" in desc:
            return self.templates["Object Acquisition"]
        if "Unlock Door" in self.templates and "door" in desc:
            return self.templates["Unlock Door"]
        if "wall" in desc or "lava" in desc or "door" in desc:
            if "Obstacle Avoidance" in self.templates:
                return self.templates["Obstacle Avoidance"]
        return self.templates["Direct Navigation"]

class ProblemDistiller:
    @staticmethod
    def distill(obs_text: str) -> str:
        return obs_text

class BufferManager:
    def __init__(self, buffer_size=2):
        self.meta_buffer = MetaBuffer(buffer_size=buffer_size)
        self.thought_history = deque(maxlen=50)

    def get_reasoning_aid(self, distilled_problem: str) -> str:
        template = self.meta_buffer.retrieve(distilled_problem)
        return f"[TEMPLATE: {template.name}]\n{template.reasoning_pattern}"


In [7]:
class MinigridTextWrapper:
    def __init__(self, env_id, render_mode=None):
        self.env = gym.make(env_id, render_mode=render_mode)
        self.env = FullyObsWrapper(self.env)
        self.action_map = {
            "turn_left": Actions.left,
            "turn_right": Actions.right,
            "forward": Actions.forward,
            "pickup": Actions.pickup,
            "drop": Actions.drop,
            "toggle": Actions.toggle,
            "done": Actions.done
        }

    def _base_env(self):
        return self.env.unwrapped

    def _find_goal_pos(self):
        base = self._base_env()
        grid = base.grid
        for x in range(grid.width):
            for y in range(grid.height):
                cell = grid.get(x, y)
                if cell is not None and getattr(cell, "type", None) == "goal":
                    return (int(x), int(y))
        return None

    def _scan_grid_objects(self):
        """Scan the full grid for notable objects and their positions."""
        base = self._base_env()
        grid = base.grid
        objects = []
        for x in range(grid.width):
            for y in range(grid.height):
                cell = grid.get(x, y)
                if cell is None:
                    continue
                obj_type = getattr(cell, 'type', None)
                if obj_type in ("key", "door", "box", "ball", "lava"):
                    state = ""
                    if obj_type == "door":
                        if getattr(cell, 'is_locked', False):
                            state = " (locked)"
                        elif getattr(cell, 'is_open', False):
                            state = " (open)"
                        else:
                            state = " (closed)"
                    color = getattr(cell, "color", "")
                    label = f"{color} {obj_type}".strip() if color else obj_type
                    objects.append(f"{label}{state} at [{x}, {y}]")
        return objects

    def _find_fallback_target(self):
        """For envs with no goal tile, find a box or ball as target."""
        base = self._base_env()
        grid = base.grid
        for x in range(grid.width):
            for y in range(grid.height):
                cell = grid.get(x, y)
                if cell is not None and getattr(cell, "type", None) in ("box", "ball"):
                    return (int(x), int(y)), getattr(cell, 'type', 'box')
        return None, None

    def get_text_obs(self, obs):
        base = self._base_env()

        if hasattr(base, "agent_pos") and base.agent_pos is not None:
            ax, ay = int(base.agent_pos[0]), int(base.agent_pos[1])
            agent_pos_text = f"[{ax}, {ay}]"
        else:
            agent_pos_text = "None"

        agent_dir = int(base.agent_dir) if hasattr(base, "agent_dir") else None
        goal_pos = self._find_goal_pos()

        if goal_pos is not None:
            target_text = f"Goal is at [{goal_pos[0]}, {goal_pos[1]}]."
        else:
            fallback_pos, fallback_type = self._find_fallback_target()
            if fallback_pos is not None:
                target_text = f"Target ({fallback_type}) is at [{fallback_pos[0]}, {fallback_pos[1]}]."
            else:
                target_text = "No target found."

        dirs = ["right", "down", "left", "up"]
        facing = dirs[agent_dir] if agent_dir is not None and 0 <= agent_dir < 4 else "unknown"
        desc = f"Agent is at {agent_pos_text} facing {facing}. {target_text}"

        front_obj = None
        if hasattr(base, "front_pos") and hasattr(base, "grid"):
            fx, fy = int(base.front_pos[0]), int(base.front_pos[1])
            front_obj = base.grid.get(fx, fy)

        if front_obj:
            desc += f" In front of you is a {front_obj.type}."
        else:
            desc += " The path in front is clear."

        grid_objects = self._scan_grid_objects()
        if grid_objects:
            desc += " Nearby objects: " + ", ".join(grid_objects) + "."

        return desc

    def reset(self):
        reset_out = self.env.reset()
        if isinstance(reset_out, tuple) and len(reset_out) == 2:
            obs, _ = reset_out
        else:
            obs = reset_out
        return self.get_text_obs(obs)

    def step(self, action_str):
        action = self.action_map.get(action_str.lower(), Actions.forward)
        obs, reward, terminated, truncated, info = self.env.step(action)
        done = terminated or truncated
        return self.get_text_obs(obs), reward, done, info


In [8]:
class LLMClient:
    def __init__(self, model_name="Qwen/Qwen2.5-3B-Instruct", max_new_tokens=16, temperature=0.0, mock=False):
        self.model_name = model_name
        self.max_new_tokens = max_new_tokens
        self.temperature = temperature
        self.mock = mock
        self.total_tokens = 0
        self.total_latency = 0
        self.model = None
        self.tokenizer = None

        if not self.mock:
            self._load_hf_model()

    def _load_hf_model(self):
        self.tokenizer = AutoTokenizer.from_pretrained(self.model_name, trust_remote_code=True)
        self.model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype="auto",
            device_map="auto",
            trust_remote_code=True,
        )
        self.model.eval()
        print(f"Loaded model: {self.model_name}")

    def query(self, system_prompt, user_prompt):
        start_time = time.time()

        if self.mock:
            response = self._mock_logic(user_prompt)
            tokens = len(system_prompt + user_prompt) // 4
        else:
            messages = [
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt},
            ]
            prompt = self.tokenizer.apply_chat_template(
                messages,
                tokenize=False,
                add_generation_prompt=True,
            )

            inputs = self.tokenizer(prompt, return_tensors="pt")
            inputs = {k: v.to(self.model.device) for k, v in inputs.items()}
            input_tokens = inputs["input_ids"].shape[-1]

            gen_kwargs = {
                "max_new_tokens": self.max_new_tokens,
                "do_sample": self.temperature > 0,
                "pad_token_id": self.tokenizer.eos_token_id,
            }
            if self.temperature > 0:
                gen_kwargs["temperature"] = self.temperature
                gen_kwargs["top_p"] = 0.9

            with torch.no_grad():
                outputs = self.model.generate(**inputs, **gen_kwargs)

            generated = outputs[0][input_tokens:]
            response = self.tokenizer.decode(generated, skip_special_tokens=True).strip()
            tokens = int(generated.shape[-1])

        latency = time.time() - start_time
        self.total_tokens += tokens
        self.total_latency += latency

        return response, tokens, latency

    @staticmethod
    def _extract_state_from_prompt(prompt):
        import re

        coord_pat = r"[\[\(]\s*(\d+)\s*,\s*(\d+)\s*[\]\)]"
        pos_match = re.search(rf"Agent is at {coord_pat} facing (\w+)", prompt)
        goal_match = re.search(rf"Goal is at {coord_pat}", prompt)
        if goal_match is None:
            goal_match = re.search(rf"Target \(\w+\) is at {coord_pat}", prompt)

        if not pos_match or not goal_match:
            return None

        ax, ay = int(pos_match.group(1)), int(pos_match.group(2))
        face = pos_match.group(3)
        gx, gy = int(goal_match.group(1)), int(goal_match.group(2))
        return ax, ay, face, gx, gy

    @staticmethod
    def _mock_policy_action(ax, ay, face, gx, gy):
        if face == "right":
            if gx > ax:
                return "forward"
            return "turn_left" if gy < ay else "turn_right"
        if face == "left":
            if gx < ax:
                return "forward"
            return "turn_right" if gy < ay else "turn_left"
        if face == "up":
            if gy < ay:
                return "forward"
            return "turn_right" if gx > ax else "turn_left"
        if face == "down":
            if gy > ay:
                return "forward"
            return "turn_left" if gx > ax else "turn_right"
        return "forward"

    def _mock_logic(self, prompt):
        state = self._extract_state_from_prompt(prompt)
        if state is None:
            return "forward"

        ax, ay, face, gx, gy = state
        return self._mock_policy_action(ax, ay, face, gx, gy)


In [9]:
class BoTAgent:
    def __init__(self, llm_client, buffer_manager, use_history=True, history_window=3):
        self.llm = llm_client
        self.buffer_manager = buffer_manager
        self.use_history = use_history
        self.history_window = history_window
        self.memory = []
        self.action_log = []

    @staticmethod
    def _parse_action(response_text):
        action = response_text.strip().lower()
        if "forward" in action:
            return "forward"
        if "left" in action:
            return "turn_left"
        if "right" in action:
            return "turn_right"
        if "toggle" in action:
            return "toggle"
        if "pickup" in action:
            return "pickup"
        if "drop" in action:
            return "drop"
        return "forward"

    def act(self, observation):
        distilled = ProblemDistiller.distill(observation)
        aid = self.buffer_manager.get_reasoning_aid(distilled)

        system_prompt = "You are a MiniGrid navigation agent. Choose exactly one action token: forward, turn_left, turn_right, toggle, pickup, drop."
        history_context = self.memory[-self.history_window:] if self.use_history else []
        user_prompt = (
            f"Current Obs: {observation}\n"
            f"{aid}\n"
            f"Recent history: {history_context}\n"
            "Return only one action token."
        )

        response, tokens, latency = self.llm.query(system_prompt, user_prompt)
        action = self._parse_action(response)

        self.memory.append((observation, action))
        self.action_log.append({"raw_llm_output": response, "parsed_action": action, "observation": observation})
        return action


In [10]:
def _run_single_episode(env, agent, max_steps):
    obs = env.reset()
    done = False
    steps = 0
    total_reward = 0.0

    while not done and steps < max_steps:
        action = agent.act(obs)
        obs, reward, done, info = env.step(action)
        total_reward += reward
        steps += 1

    success = total_reward > 0
    return success, steps, total_reward


def _build_episode_row(
    env_id,
    episode,
    policy,
    use_history,
    history_window,
    max_steps,
    model_name,
    mock,
    success,
    steps,
    reward,
    tokens,
    latency,
    buffer_size,
):
    return {
        "env": env_id,
        "episode": episode,
        "policy": policy,
        "use_history": use_history,
        "history_window": history_window if use_history else 0,
        "max_steps": max_steps,
        "model": model_name,
        "mock": mock,
        "buffer_size": buffer_size,
        "success": success,
        "steps": steps,
        "reward": reward,
        "tokens": tokens,
        "time": latency,
    }


def run_experiment(
    env_id,
    llm_client,
    n_episodes=10,
    max_steps=100,
    use_history=True,
    history_window=3,
    policy_label=None,
    buffer_size=2,
):
    results = []
    all_action_logs = []
    buffer_mgr = BufferManager(buffer_size=buffer_size)
    env = MinigridTextWrapper(env_id)

    policy = policy_label or ("history" if use_history else "reactive")
    print(
        f"Running {n_episodes} episodes on {env_id} | policy={policy} | "
        f"max_steps={max_steps} | model={llm_client.model_name} | mock={llm_client.mock} | buffer_size={buffer_size}"
    )

    for i in range(n_episodes):
        agent = BoTAgent(
            llm_client,
            buffer_mgr,
            use_history=use_history,
            history_window=history_window,
        )

        tokens_before = llm_client.total_tokens
        latency_before = llm_client.total_latency

        success, steps, total_reward = _run_single_episode(
            env=env,
            agent=agent,
            max_steps=max_steps,
        )

        ep_tokens = llm_client.total_tokens - tokens_before
        ep_latency = llm_client.total_latency - latency_before

        for step_idx, log_entry in enumerate(agent.action_log):
            log_entry["env"] = env_id
            log_entry["episode"] = i
            log_entry["step"] = step_idx
            log_entry["model"] = llm_client.model_name
            log_entry["buffer_size"] = buffer_size
        all_action_logs.extend(agent.action_log)

        results.append(
            _build_episode_row(
                env_id=env_id,
                episode=i,
                policy=policy,
                use_history=use_history,
                history_window=history_window,
                max_steps=max_steps,
                model_name=llm_client.model_name,
                mock=llm_client.mock,
                success=success,
                steps=steps,
                reward=total_reward,
                tokens=ep_tokens,
                latency=ep_latency,
                buffer_size=buffer_size,
            )
        )
        print(f"  Ep {i}: Success={success}, Steps={steps}, Reward={total_reward:.3f}")

    return pd.DataFrame(results), pd.DataFrame(all_action_logs)


def _resolve_env_id(candidates):
    """Return the first available env_id from a candidate list."""
    for env_id in candidates:
        try:
            test_env = gym.make(env_id)
            test_env.close()
            return env_id
        except Exception:
            continue
    return None


def _default_env_candidate_map():
    return {
        "MiniGrid-LavaGapS7-v0": ["MiniGrid-LavaGapS7-v0"],
        "MiniGrid-BlockedUnlockPickup-v0": [
            "MiniGrid-BlockedUnlockPickup-v0",
            "MiniGrid-UnlockPickup-v0",
        ],
        "MiniGrid-DoorKey-5x5-v0": ["MiniGrid-DoorKey-5x5-v0"],
        "MiniGrid-Empty-8x8-v0": ["MiniGrid-Empty-8x8-v0"],
    }


def _resolve_environment_pairs(environments, env_candidate_map=None):
    env_candidate_map = env_candidate_map or _default_env_candidate_map()
    resolved_envs = []

    for env_name in environments:
        candidates = env_candidate_map.get(env_name, [env_name])
        resolved = _resolve_env_id(candidates)
        if resolved is None:
            print(f"Skipping {env_name}: no available env id from {candidates}")
            continue
        resolved_envs.append((env_name, resolved))

    return resolved_envs


def _empty_errors_df():
    return pd.DataFrame(
        columns=["model", "env", "resolved_env", "n_episodes_setting", "max_steps", "buffer_size", "error"]
    )


def _build_summary_frame(all_results):
    if all_results.empty:
        return pd.DataFrame(
            columns=["model", "env", "n_episodes_setting", "max_steps", "buffer_size", "success", "steps", "reward", "tokens", "time"]
        )

    return all_results.groupby(
        ["model", "env", "n_episodes_setting", "max_steps", "buffer_size"], as_index=False
    )[["success", "steps", "reward", "tokens", "time"]].mean()


def _run_single_condition(
    llm_client,
    canonical_name,
    resolved_env_id,
    n_episodes,
    max_steps,
    history_window,
    policy_label,
    buffer_size,
):
    try:
        df, action_logs = run_experiment(
            env_id=resolved_env_id,
            llm_client=llm_client,
            n_episodes=n_episodes,
            max_steps=max_steps,
            use_history=True,
            history_window=history_window,
            policy_label=policy_label,
            buffer_size=buffer_size,
        )
        df["env"] = canonical_name
        df["resolved_env"] = resolved_env_id
        df["n_episodes_setting"] = int(n_episodes)
        return df, action_logs, None
    except Exception as e:
        error_row = {
            "model": llm_client.model_name,
            "env": canonical_name,
            "resolved_env": resolved_env_id,
            "n_episodes_setting": int(n_episodes),
            "max_steps": int(max_steps),
            "buffer_size": int(buffer_size),
            "error": str(e),
        }
        return None, None, error_row


def build_run_config(
    model_candidates,
    mock_mode,
    max_new_tokens,
    temperature,
    history_window,
    environments,
    n_episodes_list,
    max_steps_list,
    buffer_sizes,
):
    return {
        "model_candidates": list(model_candidates),
        "mock_mode": bool(mock_mode),
        "max_new_tokens": int(max_new_tokens),
        "temperature": float(temperature),
        "history_window": int(history_window),
        "environments": list(environments),
        "n_episodes_list": list(n_episodes_list),
        "max_steps_list": list(max_steps_list),
        "buffer_sizes": list(buffer_sizes),
    }


def run_bot_experiments(
    model_names,
    environments,
    n_episodes_list,
    max_steps_list,
    buffer_sizes=(2,),
    history_window=3,
    mock=True,
    max_new_tokens=16,
    temperature=0.0,
    return_errors=False,
):
    resolved_envs = _resolve_environment_pairs(environments)

    frames = []
    all_action_logs = []
    errors = []

    total_runs = len(model_names) * len(resolved_envs) * len(n_episodes_list) * len(max_steps_list) * len(buffer_sizes)
    current_run = 0

    for model_name in model_names:
        llm_client = LLMClient(
            model_name=model_name,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            mock=mock,
        )

        for canonical_name, resolved_env_id in resolved_envs:
            for n_episodes in n_episodes_list:
                for max_steps in max_steps_list:
                    for buffer_size in buffer_sizes:
                        policy_label = f"BoT_buf{buffer_size}_k{history_window}"
                        current_run += 1
                        print(
                            f"\n[{current_run}/{total_runs}] {model_name} | {canonical_name} "
                            f"| n_ep={n_episodes} | max_steps={max_steps} | buf={buffer_size}"
                        )

                        condition_df, action_logs, error_row = _run_single_condition(
                            llm_client=llm_client,
                            canonical_name=canonical_name,
                            resolved_env_id=resolved_env_id,
                            n_episodes=n_episodes,
                            max_steps=max_steps,
                            history_window=history_window,
                            policy_label=policy_label,
                            buffer_size=buffer_size,
                        )

                        if condition_df is not None:
                            frames.append(condition_df)
                        if action_logs is not None and not action_logs.empty:
                            all_action_logs.append(action_logs)
                        if error_row is not None:
                            errors.append(error_row)
                            print(f"Error: {error_row['error']}")

    all_results = pd.concat(frames, ignore_index=True) if frames else pd.DataFrame()
    combined_action_logs = pd.concat(all_action_logs, ignore_index=True) if all_action_logs else pd.DataFrame()
    errors_df = pd.DataFrame(errors) if errors else _empty_errors_df()

    if return_errors:
        return all_results, combined_action_logs, errors_df
    return all_results, combined_action_logs


def save_run_outputs(
    all_results,
    action_logs=None,
    run_errors=None,
    output_root="CS 228/outputs",
    run_tag="bot_sweep",
    config=None,
):
    """Minimal structured save: episodes/summary/actions/errors + config/metadata."""
    run_id = f"{datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')}_{run_tag}"
    run_dir = Path(output_root) / run_id
    run_dir.mkdir(parents=True, exist_ok=True)

    episodes_path = run_dir / "episodes.csv"
    summary_path = run_dir / "summary.csv"
    actions_path = run_dir / "actions.csv"
    errors_path = run_dir / "errors.csv"
    config_path = run_dir / "config.json"
    metadata_path = run_dir / "metadata.json"

    all_results.to_csv(episodes_path, index=False)
    summary = _build_summary_frame(all_results)
    summary.to_csv(summary_path, index=False)

    if action_logs is not None and not action_logs.empty:
        action_logs.to_csv(actions_path, index=False)

    if run_errors is None:
        run_errors = _empty_errors_df()
    run_errors.to_csv(errors_path, index=False)

    if config is None:
        config = {}
    with config_path.open("w", encoding="utf-8") as f:
        json.dump(config, f, indent=2)

    metadata = {
        "run_id": run_id,
        "saved_at": datetime.now(timezone.utc).isoformat(),
        "rows": int(len(all_results)),
        "action_log_rows": int(len(action_logs)) if action_logs is not None else 0,
        "errors": int(len(run_errors)),
    }
    with metadata_path.open("w", encoding="utf-8") as f:
        json.dump(metadata, f, indent=2)

    return {
        "run_id": run_id,
        "run_dir": str(run_dir),
        "episodes_path": str(episodes_path),
        "summary_path": str(summary_path),
        "actions_path": str(actions_path),
        "errors_path": str(errors_path),
        "config_path": str(config_path),
        "metadata_path": str(metadata_path),
    }


In [11]:
all_results, action_logs, run_errors = run_bot_experiments(
    model_names=MODEL_CANDIDATES,
    environments=ENVIRONMENTS,
    n_episodes_list=N_EPISODES_LIST,
    max_steps_list=MAX_STEPS_LIST,
    buffer_sizes=BUFFER_SIZES,
    history_window=HISTORY_WINDOW,
    mock=MOCK_MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    return_errors=True,
)

run_config = build_run_config(
    model_candidates=MODEL_CANDIDATES,
    mock_mode=MOCK_MODE,
    max_new_tokens=MAX_NEW_TOKENS,
    temperature=TEMPERATURE,
    history_window=HISTORY_WINDOW,
    environments=ENVIRONMENTS,
    n_episodes_list=N_EPISODES_LIST,
    max_steps_list=MAX_STEPS_LIST,
    buffer_sizes=BUFFER_SIZES,
)

run_artifacts = save_run_outputs(
    all_results=all_results,
    action_logs=action_logs,
    run_errors=run_errors,
    output_root=OUTPUT_ROOT,
    run_tag=RUN_TAG,
    config=run_config,
)

print("\nExperiment Results Summary")
print(f"Rows: {len(all_results)}")
print(f"Action log rows: {len(action_logs)}")
print(f"Errors: {len(run_errors)}")
print(f"Saved to: {run_artifacts['run_dir']}")

display(all_results.head(10))
if not run_errors.empty:
    display(run_errors.head(10))


In [ ]:
def _save_displayed_figure(fig, artifacts):
    if not artifacts:
        return None
    out_path = Path(artifacts["run_dir"]) / "overview.png"
    fig.savefig(out_path, dpi=150, bbox_inches="tight")
    return out_path


def _make_label(row):
    return f"{row[0]}\nbuf={row[1]}"


def _build_plot_inputs(all_results):
    df = all_results.copy()
    df["config"] = df["model"] + " | buf=" + df["buffer_size"].astype(str)
    success_pivot = df.groupby(["env", "config"])["success"].mean().unstack(fill_value=0)
    steps_pivot = df.groupby(["env", "config"])["steps"].mean().unstack(fill_value=0)
    buf_pivot = df.groupby(["buffer_size", "env"])["success"].mean().unstack(fill_value=0)
    env_buf_pivot = df.groupby(["env", "buffer_size"])["success"].mean().unstack(fill_value=0)
    return success_pivot, steps_pivot, buf_pivot, env_buf_pivot


def _draw_overview_figure(success_pivot, steps_pivot, buf_pivot, env_buf_pivot):
    fig, axes = plt.subplots(2, 2, figsize=(18, 12))

    success_pivot.plot(kind="bar", ax=axes[0, 0])
    axes[0, 0].set_title("Success Rate by Environment and Config")
    axes[0, 0].set_ylabel("Success Rate")
    axes[0, 0].set_ylim(0, 1)
    axes[0, 0].tick_params(axis='x', rotation=45)

    steps_pivot.plot(kind="bar", ax=axes[0, 1])
    axes[0, 1].set_title("Average Steps by Environment and Config")
    axes[0, 1].set_ylabel("Steps")
    axes[0, 1].tick_params(axis='x', rotation=45)

    buf_pivot.plot(kind="bar", ax=axes[1, 0])
    axes[1, 0].set_title("Success Rate: Buffer Size 2 vs 5")
    axes[1, 0].set_xlabel("Buffer Size")
    axes[1, 0].set_ylabel("Success Rate")
    axes[1, 0].set_ylim(0, 1)

    env_buf_pivot.plot(kind="bar", ax=axes[1, 1])
    axes[1, 1].set_title("Success Rate by Environment (grouped by buffer_size)")
    axes[1, 1].set_ylabel("Success Rate")
    axes[1, 1].set_ylim(0, 1)
    axes[1, 1].tick_params(axis='x', rotation=45)

    plt.tight_layout()
    return fig


if "all_results" not in dir() or all_results.empty:
    print("No experiment results available to visualize.")
else:
    displayed_plot_paths = []

    success_pivot, steps_pivot, buf_pivot, env_buf_pivot = _build_plot_inputs(all_results)
    fig = _draw_overview_figure(success_pivot, steps_pivot, buf_pivot, env_buf_pivot)

    artifacts = globals().get("run_artifacts")
    saved_path = _save_displayed_figure(fig, artifacts)
    if saved_path is not None:
        displayed_plot_paths.append(str(saved_path))

    plt.show()

    if displayed_plot_paths:
        print("Saved displayed plots:")
        for p in displayed_plot_paths:
            print(f"- {p}")

    summary = all_results.groupby(["model", "env", "n_episodes_setting", "max_steps", "buffer_size"])[
        ["success", "steps", "reward", "tokens", "time"]
    ].mean()
    display(summary)
